# 01 预处理：国债 ETF 与债券 snapshot 高频数据

本 Notebook 实现可复用、可审计的数据清洗 Pipeline，包括：

1. 文件加载与元数据收集
2. Schema diagnostics
3. trade_time diagnostics
4. 重复文件检测（含 hash）
5. 单文件清洗（时间处理、数值转换、缺失值填充、winsorization、市场规则检查、交易阶段分析）
6. 批量清洗
7. merge_asof 时间对齐
8. 合并后诊断
9. 结果保存（cleaned data + 各类 report）


In [ ]:
from __future__ import annotations
from pathlib import Path
from typing import Any
import hashlib
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)


In [ ]:
# =========================
# 1) 配置
# =========================
BASE_DIRS = {
    '511090': Path('国债ETF数据/3秒快照/511090'),
    '511130': Path('国债ETF数据/3秒快照/511130'),
    '019776': Path('债券数据/3秒快照/019776'),
    '019742': Path('债券数据/3秒快照/019742'),
    '019789': Path('债券数据/3秒快照/019789'),
}

CONFIG = {
    'time_col': 'trade_time',
    'id_cols': ['code'],
    'category_cols': ['trading_phase_code'],
    'drop_unnamed_index_col': True,
    'winsor_lower_q': 0.05,
    'winsor_upper_q': 0.95,
    'merge_tolerance': '1s',
    'merge_direction': 'nearest',
    'FILTER_CONTINUOUS_TRADING': False,
    'ALLOWED_TRADING_PHASES': None,
    'save_outputs': True,
}

PRICE_KEYWORDS = ['price', 'open', 'high', 'low', 'close', 'last', 'pre_close', 'iopv', 'limited']
VOLUME_KEYWORDS = ['volume', 'amount', 'num_trades']

output_clean_dir = Path('cleaned_single_files')
output_report_dir = Path('reports')
output_clean_dir.mkdir(parents=True, exist_ok=True)
output_report_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
# =========================
# 2) 文件元数据与文件名列表
# =========================
all_paths = {k: sorted(v.glob('*.csv')) for k, v in BASE_DIRS.items()}

meta_records = []
for key, paths in all_paths.items():
    print(f"\n目录 {key} - 文件数量: {len(paths)}")
    for p in paths:
        print('  -', p.name)
        meta_records.append({
            'folder': key,
            'file_name': p.name,
            'path': str(p),
            'exists': p.exists(),
            'file_size_bytes': p.stat().st_size if p.exists() else np.nan,
        })

file_meta_df = pd.DataFrame(meta_records)
file_meta_df.head()


In [ ]:
# =========================
# 3) 预分析 trade_time 参考表（按题目给定）
# =========================
pre_analysis_records = [
    # 511090
    {'folder':'511090','file_name':'511090.SH_20250701_20250731_snapshot.csv','trade_time_min':'2025-07-01 09:00:04','trade_time_max':'2025-07-31 15:59:36','trade_time_nat_count':0,'row_count':123037},
    {'folder':'511090','file_name':'511090.SH_20250801_20250831_snapshot.csv','trade_time_min':'2025-08-01 09:00:15','trade_time_max':'2025-08-29 15:59:48','trade_time_nat_count':0,'row_count':113160},
    {'folder':'511090','file_name':'511090.SH_20250901_20250930_snapshot(1).csv','trade_time_min':'2025-09-01 09:00:09','trade_time_max':'2025-09-30 15:59:31','trade_time_nat_count':0,'row_count':118378},
    {'folder':'511090','file_name':'511090.SH_20250901_20250930_snapshot.csv','trade_time_min':'2025-09-01 09:00:09','trade_time_max':'2025-09-30 15:59:31','trade_time_nat_count':0,'row_count':118378},
    {'folder':'511090','file_name':'511090.SH_20251001_20251031_snapshot(1).csv','trade_time_min':'2025-10-09 09:00:17','trade_time_max':'2025-10-31 15:59:44','trade_time_nat_count':0,'row_count':91059},
    {'folder':'511090','file_name':'511090.SH_20251001_20251031_snapshot.csv','trade_time_min':'2025-10-09 09:00:17','trade_time_max':'2025-10-31 15:59:44','trade_time_nat_count':0,'row_count':91059},
    {'folder':'511090','file_name':'511090.SH_20251101_20251130_snapshot(1).csv','trade_time_min':'2025-11-03 09:00:03','trade_time_max':'2025-11-28 15:59:44','trade_time_nat_count':0,'row_count':106779},
    {'folder':'511090','file_name':'511090.SH_20251101_20251130_snapshot.csv','trade_time_min':'2025-11-03 09:00:03','trade_time_max':'2025-11-28 15:59:44','trade_time_nat_count':0,'row_count':106779},
    {'folder':'511090','file_name':'511090.SH_20251201_20251231_snapshot.csv','trade_time_min':'2025-12-01 09:00:11','trade_time_max':'2025-12-31 15:59:38','trade_time_nat_count':0,'row_count':122430},
    # 511130
    {'folder':'511130','file_name':'511130.SH_20260101_20260131_snapshot(1).csv','trade_time_min':'2026-01-05 09:00:14','trade_time_max':'2026-01-30 16:29:54','trade_time_nat_count':0,'row_count':106236},
    {'folder':'511130','file_name':'511130.SH_20260101_20260131_snapshot.csv','trade_time_min':'2026-01-05 09:00:14','trade_time_max':'2026-01-30 16:29:54','trade_time_nat_count':0,'row_count':106236},
    {'folder':'511130','file_name':'511130.SH_20260201_20260228_snapshot.csv','trade_time_min':'2026-02-02 09:00:25','trade_time_max':'2026-02-27 16:29:56','trade_time_nat_count':0,'row_count':74345},
    # 019776
    {'folder':'019776','file_name':'019776.SH_20260101_20260131_snapshot.csv','trade_time_min':'2026-01-05 09:00:20','trade_time_max':'2026-01-30 16:29:36','trade_time_nat_count':0,'row_count':112407},
    {'folder':'019776','file_name':'019776.SH_20260201_20260228_snapshot.csv','trade_time_min':'2026-02-02 09:00:25','trade_time_max':'2026-02-27 16:29:35','trade_time_nat_count':0,'row_count':76967},
    {'folder':'019776','file_name':'019776.SH_20260301_20260312_snapshot.csv','trade_time_min':'2026-03-02 09:00:21','trade_time_max':'2026-03-12 16:29:35','trade_time_nat_count':0,'row_count':50068},
    # 019742
    {'folder':'019742','file_name':'019742.SH_20260101_20260131_snapshot.csv','trade_time_min':'2026-01-05 09:00:00','trade_time_max':'2026-01-30 16:29:37','trade_time_nat_count':0,'row_count':113148},
    {'folder':'019742','file_name':'019742.SH_20260201_20260228_snapshot.csv','trade_time_min':'2026-02-02 09:00:26','trade_time_max':'2026-02-27 16:29:45','trade_time_nat_count':0,'row_count':77589},
    {'folder':'019742','file_name':'019742.SH_20260301_20260312_snapshot.csv','trade_time_min':'2026-03-02 09:00:05','trade_time_max':'2026-03-12 16:29:48','trade_time_nat_count':0,'row_count':49949},
    # 019789
    {'folder':'019789','file_name':'019789.SH_20260101_20260131_snapshot.csv','trade_time_min':'2026-01-05 09:00:10','trade_time_max':'2026-01-30 16:29:43','trade_time_nat_count':0,'row_count':114350},
    {'folder':'019789','file_name':'019789.SH_20260201_20260228_snapshot.csv','trade_time_min':'2026-02-02 09:00:00','trade_time_max':'2026-02-27 16:30:00','trade_time_nat_count':0,'row_count':77716},
    {'folder':'019789','file_name':'019789.SH_20260301_20260312_snapshot.csv','trade_time_min':'2026-03-02 09:00:16','trade_time_max':'2026-03-12 16:29:56','trade_time_nat_count':0,'row_count':50530},
]
pre_analysis_df = pd.DataFrame(pre_analysis_records)
pre_analysis_df


In [ ]:
# =========================
# 4) 工具函数定义
# =========================
def read_snapshot_csv(path: str | Path) -> pd.DataFrame:
    return pd.read_csv(path)

def normalize_trade_time(df: pd.DataFrame, time_col: str = 'trade_time'):
    out = df.copy()
    diag = {'time_col_exists': time_col in out.columns}
    if time_col not in out.columns:
        diag.update({'nat_count': None, 'dropped_nat_rows':0, 'duplicate_time_count':None, 'is_monotonic_after_sort':None})
        return out, diag
    raw_len = len(out)
    out[time_col] = pd.to_datetime(out[time_col], errors='coerce')
    nat_count = int(out[time_col].isna().sum())
    out = out.dropna(subset=[time_col]).copy()
    dropped_nat_rows = raw_len - len(out)
    out = out.sort_values(time_col)
    dup_subset = ['code', time_col] if 'code' in out.columns else [time_col]
    duplicate_time_count = int(out.duplicated(subset=dup_subset).sum())
    out = out.drop_duplicates(subset=dup_subset, keep='last').copy()
    diag.update({'nat_count':nat_count,'dropped_nat_rows':dropped_nat_rows,'duplicate_time_count':duplicate_time_count,'is_monotonic_after_sort':bool(out[time_col].is_monotonic_increasing),'trade_time_min':out[time_col].min() if len(out)>0 else pd.NaT,'trade_time_max':out[time_col].max() if len(out)>0 else pd.NaT})
    return out, diag

def infer_column_groups(df: pd.DataFrame) -> dict:
    cols = list(df.columns)
    time_cols = [c for c in cols if 'time' in c.lower()]
    id_cols = [c for c in cols if c in ['code'] or c.endswith('_id')]
    cat_cols = [c for c in cols if c in ['trading_phase_code']]
    numeric_like = [c for c in cols if c not in set(time_cols+id_cols+cat_cols)]
    price_cols = [c for c in numeric_like if any(k in c.lower() for k in PRICE_KEYWORDS)]
    volume_cols = [c for c in numeric_like if any(k in c.lower() for k in VOLUME_KEYWORDS)]
    quote_cols = [c for c in numeric_like if ('bid_' in c.lower() or 'ask_' in c.lower())]
    return {'time_cols':time_cols,'id_cols':id_cols,'categorical_cols':cat_cols,'numeric_candidate_cols':numeric_like,'price_cols':price_cols,'volume_cols':volume_cols,'quote_cols':quote_cols}

def coerce_numeric_columns(df: pd.DataFrame, exclude_cols: list[str]):
    out = df.copy()
    report = []
    for c in out.columns:
        if c in exclude_cols:
            continue
        before_na = out[c].isna().sum()
        conv = pd.to_numeric(out[c], errors='coerce')
        if pd.api.types.is_numeric_dtype(conv):
            out[c] = conv
            after_na = out[c].isna().sum()
            report.append({'column':c,'missing_before':int(before_na),'missing_after':int(after_na),'new_nan_from_coerce':int(after_na-before_na)})
    return out, pd.DataFrame(report)

def compute_missing_report(df_before: pd.DataFrame, df_after: pd.DataFrame | None = None):
    rec=[]
    n_before = max(len(df_before),1)
    for c in df_before.columns:
        rec.append({'column':c,'missing_count_before':int(df_before[c].isna().sum()),'missing_ratio_before':float(df_before[c].isna().mean())})
    out = pd.DataFrame(rec)
    if df_after is not None:
        n_after=max(len(df_after),1)
        out['missing_count_after'] = out['column'].map(df_after.isna().sum().to_dict()).fillna(0).astype(int)
        out['missing_ratio_after'] = out['column'].map((df_after.isna().sum()/n_after).to_dict()).fillna(0.0)
        out['filled_count'] = out['missing_count_before'] - out['missing_count_after']
    return out

def compute_summary_statistics(df: pd.DataFrame, numeric_cols: list[str]):
    if len(numeric_cols)==0:
        return pd.DataFrame()
    sub = df[numeric_cols]
    s = sub.agg(['mean','std','min','max','median']).T
    s['q05'] = sub.quantile(0.05)
    s['q75'] = sub.quantile(0.75)
    s['q95'] = sub.quantile(0.95)
    s['missing_count'] = sub.isna().sum()
    s['missing_ratio'] = sub.isna().mean()
    s = s.reset_index().rename(columns={'index':'column'})
    return s

def winsorize_numeric_columns(df: pd.DataFrame, cols: list[str], lower_q=0.05, upper_q=0.95):
    out = df.copy(); rec=[]
    for c in cols:
        if c not in out.columns or not pd.api.types.is_numeric_dtype(out[c]):
            continue
        lower = out[c].quantile(lower_q); upper = out[c].quantile(upper_q)
        below = int((out[c] < lower).sum()); above = int((out[c] > upper).sum())
        out[c] = out[c].clip(lower=lower, upper=upper)
        rec.append({'column':c,'lower_bound':lower,'upper_bound':upper,'below_lower_count':below,'above_upper_count':above,'clipped_total_count':below+above})
    return out, pd.DataFrame(rec)

def run_market_sanity_checks(df: pd.DataFrame):
    rec=[]; n=max(len(df),1)
    def add(name, mask=None, skipped=False):
        if skipped:
            rec.append({'check':name,'skipped':True,'abnormal_count':None,'abnormal_ratio':None}); return
        cnt = int(mask.sum()) if mask is not None else 0
        rec.append({'check':name,'skipped':False,'abnormal_count':cnt,'abnormal_ratio':cnt/n})
    price_cols = [c for c in df.columns if any(k in c.lower() for k in ['open','high','low','close','last','bid_price','ask_price'])]
    if price_cols:
        for c in price_cols: add(f'price_positive::{c}', mask=(df[c] <= 0) & df[c].notna())
    else: add('price_positive', skipped=True)
    if all(c in df.columns for c in ['high','low']): add('high_ge_low', mask=(df['high'] < df['low']) & df['high'].notna() & df['low'].notna())
    else: add('high_ge_low', skipped=True)
    if all(c in df.columns for c in ['open','close','high','low']):
        add('open_in_range', mask=((df['open']<df['low'])|(df['open']>df['high'])) & df['open'].notna())
        add('close_in_range', mask=((df['close']<df['low'])|(df['close']>df['high'])) & df['close'].notna())
    else:
        add('open_in_range', skipped=True); add('close_in_range', skipped=True)
    for c in ['volume','amount','num_trades']:
        if c in df.columns: add(f'nonnegative::{c}', mask=(df[c] < 0) & df[c].notna())
        else: add(f'nonnegative::{c}', skipped=True)
    if all(c in df.columns for c in ['bid_price1','ask_price1']): add('bid_le_ask_l1', mask=(df['bid_price1']>df['ask_price1']) & df['bid_price1'].notna() & df['ask_price1'].notna())
    else: add('bid_le_ask_l1', skipped=True)
    for c in [x for x in df.columns if 'bid_volume' in x.lower() or 'ask_volume' in x.lower()]:
        add(f'nonnegative::{c}', mask=(df[c] < 0) & df[c].notna())
    if all(c in df.columns for c in ['high_limited','low_limited','high','low','close']):
        add('high_le_high_limited', mask=(df['high'] > df['high_limited']) & df['high'].notna() & df['high_limited'].notna())
        add('low_ge_low_limited', mask=(df['low'] < df['low_limited']) & df['low'].notna() & df['low_limited'].notna())
        add('close_within_limits', mask=((df['close'] > df['high_limited']) | (df['close'] < df['low_limited'])) & df['close'].notna())
    else:
        add('high_le_high_limited', skipped=True); add('low_ge_low_limited', skipped=True); add('close_within_limits', skipped=True)
    return pd.DataFrame(rec)

def analyze_trading_phase(df: pd.DataFrame):
    if 'trading_phase_code' not in df.columns:
        return pd.DataFrame([{'trading_phase_code':'<MISSING_COL>','count':None,'ratio':None}])
    vc = df['trading_phase_code'].value_counts(dropna=False)
    out = vc.rename_axis('trading_phase_code').reset_index(name='count')
    out['ratio'] = out['count']/max(len(df),1)
    return out

def file_sha256(path: Path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()


In [ ]:
# =========================
# 5) Schema + trade_time + duplicate file 预诊断
# =========================
schema_records=[]
trade_time_records=[]
dup_base=[]

for folder, paths in all_paths.items():
    for p in paths:
        df = read_snapshot_csv(p)
        cols = list(df.columns)
        schema_records.append({
            'folder':folder,'file_name':p.name,'row_count':len(df),'col_count':len(cols),
            'all_columns':'|'.join(cols),
            'missing_key_columns':'|'.join([c for c in ['trade_time','code'] if c not in cols]),
            'has_unnamed_0':('Unnamed: 0' in cols),
        })
        if 'trade_time' in cols:
            tt = pd.to_datetime(df['trade_time'], errors='coerce')
            trade_time_records.append({'folder':folder,'file_name':p.name,'trade_time_min':tt.min(),'trade_time_max':tt.max(),'trade_time_nat_count':int(tt.isna().sum()),'row_count':len(df)})
        else:
            trade_time_records.append({'folder':folder,'file_name':p.name,'trade_time_min':pd.NaT,'trade_time_max':pd.NaT,'trade_time_nat_count':None,'row_count':len(df)})
        dup_base.append({'folder':folder,'file_name':p.name,'path':str(p),'file_size_bytes':p.stat().st_size,'row_count':len(df),'trade_time_min':trade_time_records[-1]['trade_time_min'],'trade_time_max':trade_time_records[-1]['trade_time_max'],'col_signature':'|'.join(cols),'sha256':file_sha256(p)})

schema_df = pd.DataFrame(schema_records)
trade_time_df = pd.DataFrame(trade_time_records)
dup_base_df = pd.DataFrame(dup_base)

# duplicate group 判断
key_cols = ['folder','row_count','trade_time_min','trade_time_max','file_size_bytes','col_signature']
dup_base_df['dup_group_id'] = dup_base_df.groupby(key_cols, dropna=False).ngroup()

actions=[]
for gid, g in dup_base_df.groupby('dup_group_id'):
    hashes = g['sha256'].nunique()
    if len(g)==1:
        for _,r in g.iterrows(): actions.append((r['file_name'],'keep','unique'))
    else:
        if hashes == 1:
            first=True
            for _,r in g.iterrows():
                actions.append((r['file_name'],'keep' if first else 'drop','exact_duplicate'))
                first=False
        else:
            for _,r in g.iterrows(): actions.append((r['file_name'],'suspicious','same_meta_diff_hash'))

action_df = pd.DataFrame(actions, columns=['file_name','action','duplicate_type'])
duplicate_file_report = dup_base_df.merge(action_df, on='file_name', how='left').sort_values(['folder','file_name'])

display(schema_df.head())
display(trade_time_df.head())
display(duplicate_file_report[['folder','file_name','dup_group_id','sha256','action','duplicate_type']].head(20))


In [ ]:
# =========================
# 6) 单文件清洗函数 + 批量清洗
# =========================
def clean_single_file(path: str | Path, config: dict):
    path = Path(path)
    raw = read_snapshot_csv(path)
    diagnostics = {'file_name':path.name,'path':str(path),'shape_before':raw.shape}

    df = raw.copy()
    if config.get('drop_unnamed_index_col', True) and 'Unnamed: 0' in df.columns:
        df = df.drop(columns=['Unnamed: 0'])
        diagnostics['dropped_unnamed_0'] = True
    else:
        diagnostics['dropped_unnamed_0'] = False

    df, tt_diag = normalize_trade_time(df, time_col=config.get('time_col','trade_time'))
    diagnostics.update(tt_diag)

    groups = infer_column_groups(df)
    exclude_cols = list(set(groups['time_cols'] + groups['id_cols'] + groups['categorical_cols']))

    df_num, num_coerce_report = coerce_numeric_columns(df, exclude_cols=exclude_cols)

    missing_before_fill = compute_missing_report(df_num)

    # 只对价格类字段做前向填充，不对 volume/amount/num_trades 填充
    price_cols = [c for c in groups['price_cols'] if c in df_num.columns]
    df_filled = df_num.copy()
    if price_cols:
        df_filled[price_cols] = df_filled[price_cols].ffill()

    missing_after_fill = compute_missing_report(df_num, df_filled)

    numeric_cols = [c for c in df_filled.columns if c not in exclude_cols and pd.api.types.is_numeric_dtype(df_filled[c])]
    summary_stats = compute_summary_statistics(df_filled, numeric_cols=numeric_cols)

    # 不对疑似二元/低基数状态列 winsorize
    winsor_cols = [c for c in numeric_cols if df_filled[c].nunique(dropna=True) > 10]
    df_win, winsor_report = winsorize_numeric_columns(df_filled, winsor_cols, config['winsor_lower_q'], config['winsor_upper_q'])

    sanity_report = run_market_sanity_checks(df_win)
    phase_report = analyze_trading_phase(df_win)

    if config.get('FILTER_CONTINUOUS_TRADING', False) and config.get('ALLOWED_TRADING_PHASES') and 'trading_phase_code' in df_win.columns:
        before = len(df_win)
        df_win = df_win[df_win['trading_phase_code'].isin(config['ALLOWED_TRADING_PHASES'])].copy()
        diagnostics['phase_filter_removed_rows'] = int(before - len(df_win))
    else:
        diagnostics['phase_filter_removed_rows'] = 0

    diagnostics['shape_after'] = df_win.shape

    pack = {
        'cleaned_df': df_win,
        'num_coerce_report': num_coerce_report,
        'missing_before_fill': missing_before_fill,
        'missing_after_fill': missing_after_fill,
        'summary_stats': summary_stats,
        'winsor_report': winsor_report,
        'sanity_report': sanity_report,
        'phase_report': phase_report,
        'diagnostics': diagnostics,
    }
    return df_win, pack

def batch_clean_files(paths: list[Path], config: dict):
    cleaned = {}
    packs = {}
    diag_records = []
    for p in paths:
        cdf, pack = clean_single_file(p, config)
        cleaned[p.name] = cdf
        packs[p.name] = pack
        diag_records.append(pack['diagnostics'])
    return cleaned, packs, pd.DataFrame(diag_records)


In [ ]:
# 按 duplicate_file_report 的 action 进行清洗（drop exact_duplicate）
selected_paths = []
for folder, paths in all_paths.items():
    for p in paths:
        action_row = duplicate_file_report.loc[duplicate_file_report['file_name']==p.name, 'action']
        action = action_row.iloc[0] if len(action_row) else 'keep'
        if action != 'drop':
            selected_paths.append(p)

cleaned_data, cleaned_packs, clean_diag_df = batch_clean_files(selected_paths, CONFIG)
clean_diag_df.head()


In [ ]:
# =========================
# 7) 合并函数与合并后诊断
# =========================
def merge_cleaned_data(cleaned_data: dict[str, pd.DataFrame], base_key: str, tolerance='1s'):
    base = cleaned_data[base_key].sort_values('trade_time').copy()
    merge_diag = []
    for k, df in cleaned_data.items():
        if k == base_key:
            continue
        right = df.sort_values('trade_time').copy()
        keep_cols = [c for c in right.columns if c not in ['trade_time']]
        rename_map = {c: f'{k}__{c}' for c in keep_cols}
        right = right[['trade_time']+keep_cols].rename(columns=rename_map)
        before_shape = base.shape
        base = pd.merge_asof(
            base.sort_values('trade_time'),
            right.sort_values('trade_time'),
            on='trade_time',
            direction=CONFIG['merge_direction'],
            tolerance=pd.Timedelta(tolerance),
        )
        merge_diag.append({'merged_with':k,'before_rows':before_shape[0],'before_cols':before_shape[1],'after_rows':base.shape[0],'after_cols':base.shape[1]})
    return base, pd.DataFrame(merge_diag)

# 选择 trade_time 覆盖更完整的文件作为 base（这里默认取最大行数）
base_key = max(cleaned_data.keys(), key=lambda x: len(cleaned_data[x]))
merged_df, merge_diag_df = merge_cleaned_data(cleaned_data, base_key=base_key, tolerance=CONFIG['merge_tolerance'])

post_merge_diag = pd.DataFrame([{
    'base_key': base_key,
    'merged_row_count': len(merged_df),
    'merged_col_count': merged_df.shape[1],
    'trade_time_min': merged_df['trade_time'].min() if 'trade_time' in merged_df.columns else pd.NaT,
    'trade_time_max': merged_df['trade_time'].max() if 'trade_time' in merged_df.columns else pd.NaT,
}])

display(merge_diag_df)
display(post_merge_diag)


In [ ]:
# =========================
# 8) 汇总并保存所有报告/结果
# =========================
# 整理跨文件报告
num_coerce_all = []
missing_report_all = []
summary_stats_all = []
winsor_all = []
sanity_all = []
phase_all = []

for fname, pack in cleaned_packs.items():
    for key, target in [
        ('num_coerce_report', num_coerce_all),
        ('missing_after_fill', missing_report_all),
        ('summary_stats', summary_stats_all),
        ('winsor_report', winsor_all),
        ('sanity_report', sanity_all),
        ('phase_report', phase_all),
    ]:
        df = pack[key].copy()
        if len(df) > 0:
            df.insert(0, 'file_name', fname)
            target.append(df)

num_coerce_all_df = pd.concat(num_coerce_all, ignore_index=True) if num_coerce_all else pd.DataFrame()
missing_value_report_df = pd.concat(missing_report_all, ignore_index=True) if missing_report_all else pd.DataFrame()
summary_statistics_df = pd.concat(summary_stats_all, ignore_index=True) if summary_stats_all else pd.DataFrame()
winsorization_report_df = pd.concat(winsor_all, ignore_index=True) if winsor_all else pd.DataFrame()
market_sanity_report_df = pd.concat(sanity_all, ignore_index=True) if sanity_all else pd.DataFrame()
trading_phase_report_df = pd.concat(phase_all, ignore_index=True) if phase_all else pd.DataFrame()

if CONFIG['save_outputs']:
    # 保存单文件清洗结果
    for fname, df in cleaned_data.items():
        out_path = output_clean_dir / f"{fname.replace('.csv','')}_cleaned.parquet"
        df.to_parquet(out_path, index=False)

    merged_df.to_parquet('merged_snapshot_data.parquet', index=False)

    schema_df.to_csv(output_report_dir / 'schema_diagnostics.csv', index=False)
    trade_time_df.to_csv(output_report_dir / 'trade_time_diagnostics.csv', index=False)
    missing_value_report_df.to_csv(output_report_dir / 'missing_value_report.csv', index=False)
    summary_statistics_df.to_csv(output_report_dir / 'summary_statistics.csv', index=False)
    winsorization_report_df.to_csv(output_report_dir / 'winsorization_report.csv', index=False)
    market_sanity_report_df.to_csv(output_report_dir / 'market_sanity_report.csv', index=False)
    duplicate_file_report.to_csv(output_report_dir / 'duplicate_file_report.csv', index=False)
    merge_diag_df.to_csv(output_report_dir / 'merge_diagnostics.csv', index=False)

print('保存完成。')
print('cleaned_single_files/ 文件数:', len(list(output_clean_dir.glob('*.parquet'))))
print('reports/ 文件数:', len(list(output_report_dir.glob('*.csv'))))
